## Initial Settings

In [ ]:
print("Starting init...")

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import time
import os
import sys
import math
import random
import numpy as np
import cv2
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
from pynq import Overlay, MMIO, PL, allocate
from pynq.lib.video import *
from pynq_dpu import DpuOverlay
import spidev
import colorsys

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

overlay = DpuOverlay("/home/xilinx/jupyter_notebooks/KGIC/driveCode/dpu/dpu.bit")
overlay.load_model("/home/xilinx/jupyter_notebooks/KGIC/modelCode/tiny-yolov3_256.xmodel")

print("Import & Model init completed.")

## 모터 초기설정

In [ ]:
# 모터 설정
motor_0_address = 0x00A0000000
motor_1_address = 0x00A0010000
motor_2_address = 0x00A0020000
motor_3_address = 0x00A0030000
motor_4_address = 0x00A0040000
motor_5_address = 0x00A0050000

address_range = 0x10000

motor_0 = MMIO(motor_0_address , address_range)
motor_1 = MMIO(motor_1_address , address_range) # CHANGE
motor_2 = MMIO(motor_2_address , address_range)
motor_3 = MMIO(motor_3_address , address_range)
motor_4 = MMIO(motor_4_address , address_range) # CHANGE
motor_5 = MMIO(motor_5_address , address_range)

In [ ]:
# 모터 기본 설정
# 300 Mhz = 3.33 ns (비바도에서 설정한 클럭 주기에 따라 달라질 수 있음. 확인 필요.)
# size = 2ms = 3.33ns x 600600.600.....
# about 600600
size = 600600
motor_0_percent = 50/100
motor_1_percent = 50/100
motor_2_percent = 50/100
motor_3_percent = 50/100
motor_4_percent = 50/100
motor_5_percent = 50/100

motor_0_duty = size * motor_0_percent
motor_1_duty = size * motor_1_percent
motor_2_duty = size * motor_2_percent
motor_3_duty = size * motor_3_percent
motor_4_duty = size * motor_4_percent
motor_5_duty = size * motor_5_percent

print("motor_0_duty:", int(motor_0_duty)) # 300300 -> 50%

motor_0.write(0x00,size)           # size
motor_0.write(0x04,int(motor_0_duty))   # DUTY
motor_0.write(0x08,0)              # valid

motor_1.write(0x00,size)  # size
motor_1.write(0x04,int(motor_1_duty))   # DUTY
motor_1.write(0x08,0)       # valid

motor_2.write(0x00,size)  # size
motor_2.write(0x04,int(motor_2_duty))   # DUTY
motor_2.write(0x08,0)       # valid

motor_3.write(0x00,size)  # size
motor_3.write(0x04,int(motor_3_duty))   # DUTY
motor_3.write(0x08,0)       # valid

motor_4.write(0x00,size)  # size
motor_4.write(0x04,int(motor_4_duty))   # DUTY
motor_4.write(0x08,0)       # valid

motor_5.write(0x00,size)  # size
motor_5.write(0x04,int(motor_5_duty))   # DUTY
motor_5.write(0x08,0)       # valid

print("Motor init completed.")

## YOLOv3 Utility functions


In [ ]:
anchor_list = [10, 14, 23, 27, 37, 58, 81, 82, 135, 169, 344, 319]
anchors = np.array(anchor_list).reshape(-1, 2)

# Reshaped anchors:
# [ [10, 14,  23,  27,  37,  58],
#   [81, 82, 135, 169, 344, 319] ]

# 클래스 정보 가져오기 함수
def get_class(classes_path):
    with open(classes_path) as f:
        class_names = f.readlines()
    class_names = [c.strip() for c in class_names]
    return class_names

# lane_class.txt
# 1

classes_path = "/home/xilinx/jupyter_notebooks/KGIC/modelCode/configs/lane_class.txt"
class_names = get_class(classes_path)

num_classes = len(class_names) # 1
hsv_tuples = [(1.0 * x / num_classes , 1. , 1.) for x in range(num_classes)] # [ (1.0, 1.0, 1.0) ]
colors = list(map(lambda x: colorsys.hsv_to_rgb(*x), hsv_tuples))
colors = list(map(lambda x:
                  (int(x[0] * 255), int(x[1] * 255), int(x[2] * 255)),
                  colors)) # denormalize
random.seed(42)
random.shuffle(colors)
random.seed(None)

# 이미지 비율을 유지하며 padding하여 리사이즈하는 함수
def letterbox_image(image, size):
    ih, iw, _ = image.shape
    w, h = size
    scale = min(w/iw, h/ih)
    nw = int(iw*scale)
    nh = int(ih*scale)
    image = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_LINEAR)
    new_image = np.ones((h, w, 3), np.uint8) * 128
    h_start = (h-nh)//2
    w_start = (w-nw)//2
    new_image[h_start:h_start+nh, w_start:w_start+nw, :] = image
    return new_image

# 이미지 전처리 함수
def pre_process(image, model_image_size):
    image = image[..., ::-1]
    image_h, image_w, _ = image.shape
    if model_image_size != (None, None):
        assert model_image_size[0] % 32 == 0, 'Multiples of 32 required'
        assert model_image_size[1] % 32 == 0, 'Multiples of 32 required'
        boxed_image = letterbox_image(image, tuple(reversed(model_image_size)))
    else:
        new_image_size = (image_w - (image_w % 32), image_h - (image_h % 32))
        boxed_image = letterbox_image(image, new_image_size)
    image_data = np.array(boxed_image, dtype='float32')
    image_data /= 255.
    image_data = np.expand_dims(image_data, 0) 	
    return image_data

def _get_feats(feats, anchors, num_classes, input_shape):
    num_anchors = len(anchors)
    anchors_tensor = np.reshape(np.array(anchors, dtype=np.float32), [1, 1, 1, num_anchors, 2])
    grid_size = np.shape(feats)[1:3]
    nu = num_classes + 5
    predictions = np.reshape(feats, [-1, grid_size[0], grid_size[1], num_anchors, nu])
    grid_y = np.tile(np.reshape(np.arange(grid_size[0]), [-1, 1, 1, 1]), [1, grid_size[1], 1, 1])
    grid_x = np.tile(np.reshape(np.arange(grid_size[1]), [1, -1, 1, 1]), [grid_size[0], 1, 1, 1])
    grid = np.concatenate([grid_x, grid_y], axis=-1)
    grid = np.array(grid, dtype=np.float32)

    # Confidence score 계산 확인
    box_xy = (1 / (1 + np.exp(-predictions[..., :2])) + grid) / np.array(grid_size[::-1], dtype=np.float32)
    box_wh = np.exp(predictions[..., 2:4]) * anchors_tensor / np.array(input_shape[::-1], dtype=np.float32)
    box_confidence = 1 / (1 + np.exp(-predictions[..., 4:5]))
    box_class_probs = 1 / (1 + np.exp(-predictions[..., 5:]))

    return box_xy, box_wh, box_confidence, box_class_probs

def correct_boxes(box_xy, box_wh, input_shape, image_shape):
    box_yx = box_xy[..., ::-1]
    box_hw = box_wh[..., ::-1]
    input_shape = np.array(input_shape, dtype = np.float32)
    image_shape = np.array(image_shape, dtype = np.float32)
    new_shape = np.around(image_shape * np.min(input_shape / image_shape))
    offset = (input_shape - new_shape) / 2. / input_shape
    scale = input_shape / new_shape
    box_yx = (box_yx - offset) * scale
    box_hw *= scale

    box_mins = box_yx - (box_hw / 2.)
    box_maxes = box_yx + (box_hw / 2.)
    boxes = np.concatenate([
        box_mins[..., 0:1],
        box_mins[..., 1:2],
        box_maxes[..., 0:1],
        box_maxes[..., 1:2]
    ], axis = -1)
    boxes *= np.concatenate([image_shape, image_shape], axis = -1)
    return boxes


def boxes_and_scores(feats, anchors, classes_num, input_shape, image_shape):
    box_xy, box_wh, box_confidence, box_class_probs = _get_feats(feats, anchors, classes_num, input_shape)
    boxes = correct_boxes(box_xy, box_wh, input_shape, image_shape)
    boxes = np.reshape(boxes, [-1, 4])
    box_scores = box_confidence * box_class_probs
    box_scores = np.reshape(box_scores, [-1, classes_num])
    return boxes, box_scores

def nms_boxes(boxes, scores):
    """Suppress non-maximal boxes.

    # Arguments
        boxes: ndarray, boxes of objects.
        scores: ndarray, scores of objects.

    # Returns
        keep: ndarray, index of effective boxes.
    """
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = (x2-x1+1)*(y2-y1+1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w1 = np.maximum(0.0, xx2 - xx1 + 1)
        h1 = np.maximum(0.0, yy2 - yy1 + 1)
        inter = w1 * h1

        ovr = inter / (areas[i] + areas[order[1:]] - inter)
        inds = np.where(ovr <= 0.1)[0]  # threshold
        order = order[inds + 1]

    return keep

# 모델 평가 함수: YOLOv3-Tiny의 2-출력 레이어 구조에 맞게 anchor_mask 설정
def evaluate(yolo_outputs, image_shape, class_names, anchors, threshold, max_boxes=20):
    score_thresh = threshold
    anchor_mask = [[3, 4, 5], [0, 1, 2]]  # YOLOv3-Tiny에 맞춘 2개의 출력 레이어용 anchor mask
    boxes = []
    box_scores = []
    input_shape = np.shape(yolo_outputs[0])[1:3] * np.array([32, 32])

    # anchors:
    # [ [10, 14,  23,  27,  37,  58],
    #   [81, 82, 135, 169, 344, 319] ]

    # anchors with anchor_mask ?
    # yolo_outputs == 1: 

    for i in range(len(yolo_outputs)):
        _boxes, _box_scores = boxes_and_scores(
            yolo_outputs[i], anchors[anchor_mask[i]], len(class_names),
            input_shape, image_shape)
        boxes.append(_boxes)
        box_scores.append(_box_scores)
    boxes = np.concatenate(boxes, axis=0)
    box_scores = np.concatenate(box_scores, axis=0)


    mask = box_scores >= score_thresh
    boxes_ = []
    scores_ = []
    classes_ = []
    for c in range(len(class_names)):
        class_boxes_np = boxes[mask[:, c]]
        class_box_scores_np = box_scores[:, c][mask[:, c]]

        # Non-Max Suppression with max_boxes limit
        nms_index_np = nms_boxes(class_boxes_np, class_box_scores_np)
        nms_index_np = nms_index_np[:max_boxes]  # Limit to max_boxes

        class_boxes_np = class_boxes_np[nms_index_np]
        class_box_scores_np = class_box_scores_np[nms_index_np]
        classes_np = np.ones_like(class_box_scores_np, dtype=np.int32) * c
        boxes_.append(class_boxes_np)
        scores_.append(class_box_scores_np)
        classes_.append(classes_np)

    boxes_ = np.concatenate(boxes_, axis=0)
    scores_ = np.concatenate(scores_, axis=0)
    classes_ = np.concatenate(classes_, axis=0)


    return boxes_, scores_, classes_


## DPU Setting

In [ ]:
# DPU: Deep Learning Processing Unit

# 1. .xmodel 파일에서 실행기(runner) 객체를 가져오기. 여기서 실제 추론이 일어남.
dpu = overlay.runner

# 2. 모델에 사전 정의된 입력/출력 텐서의 구조와 타입을 가져옴.
# tiny_yolov는 입력 1개, 출력 2개(레이어) 구조.
inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

# 3. 가져온 텐서의 shape을 출력.
# 13 * 13은 큰 객체 (coarse grid), 26 * 26은 작은 객체 (fine grid) 검출용.
# 배치란? 한 번에 이미지 몇 장을 처리할 것이냐는 뜻. 
# 자율주행은 이미지 한 장씩 처리하지만 병렬 처리의 효율화를 위해 1 이상의 배치를 사용하는 경우가 많음.
shapeIn = tuple(inputTensors[0].dims) # (1, 256, 256, 3) -> (배치, 높이, 폭, 채널(RGB))

if shapeIn != (1, 256, 256, 3):
    logger.warning("inputTensors의 shape이 예상과 다름. 예상: (1, 256, 256, 3), 실제: %s", shapeIn)

shapeOut0 = (tuple(outputTensors[0].dims)) # (1, 13, 13, 75)
shapeOut1 = (tuple(outputTensors[1].dims)) # (1, 26, 26, 75)

# 4. (?) 출력 텐서 전체 크기를 배치 크기(shapeIn[0], 보통 1)로 나눠서 1개 이미지 당 출력 원소 개수 계산.
# 여기 말고는 쓰이는 곳이 없는데 지워도 될 듯? 아마도 디버깅 용.
outputSize0 = int(outputTensors[0].get_data_size() / shapeIn[0]) # 12675 (= 13 * 13 * 75)
outputSize1 = int(outputTensors[1].get_data_size() / shapeIn[0]) # 50700 (= 26 * 26 * 75)

# 5. DPU 추론 용 버퍼 할당.
# np.empty라 초기화가 없어 소폭 빠르다. C-style buffer (행 우선 방식, 버퍼에 행 단위로 직렬화되어있다는 뜻)
input_data = [np.empty(shapeIn, dtype=np.float32, order="C")]
output_data = [np.empty(shapeOut0, dtype=np.float32, order="C"),
               np.empty(shapeOut1, dtype=np.float32, order="C")]

# image는 input_data[0]의 단순 alias. 같은 주소를 가리킴.
image = input_data[0]

## Flight Recorder (블랙박스)

In [ ]:
# ===== 플라이트 레코더 (블랙박스): 주행 데이터를 보드 로컬 디스크에 기록 =====
# 목적: 주행 문제가 인식(YOLO)/판단(각도 변환)/제어(폐루프) 중 어느 레이어인지 사후 분석.
# 설계: 주행 루프는 큐에 넣기만 하고(논블로킹), JPEG 인코딩/디스크 쓰기는 별도 쓰레드가 수행
#       -> 기록이 비전/제어 주기를 방해하지 않음. 큐가 가득 차면 해당 프레임은 버림(주행 우선).
# 분석: 주행 후 run_* 폴더를 맥으로 복사해서 replay_viewer.html 로 열기.
#   scp -r xilinx@<보드IP>:/home/xilinx/jupyter_notebooks/KGIC/driveCode/logs/run_* ~/AIchip/logs/
import json
import queue
import threading
from datetime import datetime

RECORD_BASE_DIR = "/home/xilinx/jupyter_notebooks/KGIC/driveCode/logs"
RECORD_JPEG_QUALITY = 70   # 낮출수록 용량 절약. 70이면 프레임당 대략 20~40KB

class FlightRecorder:
    def __init__(self, base_dir=RECORD_BASE_DIR, jpeg_quality=RECORD_JPEG_QUALITY):
        self.run_dir = os.path.join(base_dir, datetime.now().strftime("run_%Y%m%d_%H%M%S"))
        os.makedirs(os.path.join(self.run_dir, "frames"), exist_ok=True)
        self.jpeg_quality = jpeg_quality
        self.q = queue.Queue(maxsize=120)   # 약 수 초 분량 버퍼
        self.dropped = 0
        self._stop = False
        self._f = open(os.path.join(self.run_dir, "telemetry.jsonl"), "w")
        self._writer = threading.Thread(target=self._write_loop, daemon=True)
        self._writer.start()
        print(f"[REC] 기록 시작: {self.run_dir}")

    def record(self, vis_img, telem):
        # 주행 루프에서 매 프레임 호출. 절대 블로킹하지 않는다.
        try:
            # vis_img는 주행 루프가 다음 프레임에서 덮어쓸 수 있는 버퍼라 복사해서 넘김
            img = None if vis_img is None else vis_img.copy()
            self.q.put_nowait((img, telem))
        except queue.Full:
            self.dropped += 1

    def _write_loop(self):
        while not (self._stop and self.q.empty()):
            try:
                img, telem = self.q.get(timeout=0.2)
            except queue.Empty:
                continue
            if img is not None:
                fname = f"frames/{telem['i']:06d}.jpg"
                cv2.imwrite(os.path.join(self.run_dir, fname), img,
                            [cv2.IMWRITE_JPEG_QUALITY, self.jpeg_quality])
                telem["frame_file"] = fname
            self._f.write(json.dumps(telem) + "\n")

    def close(self):
        self._stop = True
        self._writer.join(timeout=10.0)
        self._f.flush()
        self._f.close()
        print(f"[REC] 기록 종료: {self.run_dir} (드롭된 프레임 {self.dropped}개)")

print("FlightRecorder ready")

## Driving Funtions

In [ ]:
import threading

# ================== 튜닝 파라미터 (트랙에서 조정) ==================
DRIVE_SPEED = 50          # 구동 속도(0~100)
AUTO_STEER_DUTY = 0.6     # 자율주행 조향 모터 고정 duty(0~1). 캐노니컬 control_mode=1 방식(램프 제거)
STEER_TRIM = -5.5         # 조향 중립 트림(실측): 물리적 직진의 mapped값. 차량 핸들이 중립일 때 센서 측정 값. 이 값을 이용해 실제 직진과 센서 상의 직진을 보정함.
REF_X = 200               # 조향 기준 x값. (직선 정렬상태에서 이미지 상에 표시되는 우측차선 중심 x좌표)
STEER_DIR = +1            # 스티어링 부호. 핸들이 거꾸로 움직이면 음수로 바꾸기.
STEER_GAIN = 20.0 / 128.0 # 화면 오프셋(±128px) -> 조향 타겟(±20)
CENTER_EMA = 0.4          # 차선중심 저역통과(0~1, 클수록 민감). 조향 지터 완화
CONTROL_HZ = 100          # 제어 스레드 주기(Hz) — data_collection과 동일
DISPLAY_EVERY = 1         # N프레임마다 1번만 화면 표시(느린 그리기로 인한 루프 지연 방지)
YOLO_THRESHOLD = 0.01     # YOLO 임게값
# ---- 블랙박스 모드 스위치 ----
RECORD_ENABLE = True      # True: FlightRecorder로 매 프레임 디스크 기록
DISPLAY_ENABLE = False    # False: 노트북 실시간 화면표시 끔 (wifi/그리기 오버헤드 제거, 기록으로 대체)
HEARTBEAT_EVERY = 30      # N프레임마다 한 줄만 상태 출력 (살아있는지 확인용)
# =====================================================================

# SPI 초기화 (조향 가변저항 피드백)
spi0 = spidev.SpiDev()
spi0.open(0, 0)
spi0.max_speed_hz = 20000000
spi0.mode = 0b00

class RobotController:
    # 구조: data_collection과 동일하게 "제어 폐루프는 빠른 별도 스레드(100Hz)"로 돌리고,
    #       카메라/DPU 루프는 목표 조향각(self.steering_angle)만 갱신한다.
    #       모터 write는 오직 제어 스레드에서만 발생(카메라 스레드와 충돌 없음).
    def __init__(self):
        self.image_processor = ImageProcessor()
        self.size = 600600  # clock count for 2ms in 300 Mhz clock

        self.left_speed = 0
        self.right_speed = 0
        self.steering_angle = 0.0     # 목표 조향각(-20~20). 카메라 루프가 갱신
        self.speed = DRIVE_SPEED

        # 조향 duty 램프 (data_collection과 동일)
        self.current_duty = self.size // 2
        self.min_duty = self.size // 2 # 300,300 -> 50%
        self.max_duty = int(self.size * 0.8) # 480,480 -> 80%
        self.duty_step = int(self.size * 0.02) # 12,012 -> 2%
        self.last_steering_time = time.time()

        # 가변저항 실측값 (data_collection 검증본)
        self.resistance_most_left = 2338
        self.resistance_most_right = 1512

        self.stop_flag = False
        self._center_ema = None       # 차선중심 EMA 상태
        self._dbg_target = 0.0        # 디버그: 최근 목표 조향각
        self._dbg_mapped = 0.0        # 디버그: 최근 실제 바퀴 위치
        self._dbg_adc = 0             # 디버그: 최근 ADC 원시값
        self._dbg_cmd = "stay"        # 디버그: 최근 조향 명령 (left/stay/right)
        self.recorder = None          # FlightRecorder (run에서 생성)
        self.init_motors()

    def init_motors(self):
        for motor in [motor_0, motor_1, motor_2, motor_3, motor_4, motor_5]:
            motor.write(0x00, self.size)
            motor.write(0x04, self.min_duty)
            motor.write(0x08, 0)

    def map_value(self, x, in_min, in_max, out_min, out_max):
        if x <= in_min:
            return out_max
        elif x >= in_max:
            return out_min
        else:
            return (in_max - x) * (out_max - out_min) / (in_max - in_min) + out_min

    def read_adc(self, spi):
        r = spi.xfer2([0x00, 0x00])
        return ((r[0] & 0x0F) << 8) | r[1]

    # ---- 조향 (data_collection 그대로: motor_4=좌, motor_5=우, duty 램프) ----
    def right(self):
        # 자율주행: 고정 duty (캐노니컬 control_mode=1). 램프 제거.
        motor_4.write(0x08, 0)
        motor_5.write(0x08, 1)
        motor_5.write(0x04, int(self.size * AUTO_STEER_DUTY))

    def left(self):
        # 자율주행: 고정 duty (캐노니컬 control_mode=1). 램프 제거.
        motor_5.write(0x08, 0)
        motor_4.write(0x08, 1)
        motor_4.write(0x04, int(self.size * AUTO_STEER_DUTY))

    def stay(self):
        self.current_duty = self.min_duty
        motor_5.write(0x08, 0)
        motor_4.write(0x08, 0)
        motor_5.write(0x04, self.current_duty)
        motor_4.write(0x04, self.current_duty)

    # ---- 구동 (data_collection 그대로: motor_2/3=좌, motor_0/1=우) ----
    def set_left_speed(self, speed):

        ## 나중에 에러에 따라 duty를 변경하는 방법도 고려 가능할 듯
        # err = abs(mapped_resistance - target)
        # duty = AUTO_STEER_DUTY if err > 3.0 else AUTO_STEER_DUTY * 0.5  # 목표 근처면 절반 힘

        duty = int(self.size * (abs(speed) / 100))
        motor_2.write(0x04, duty)
        motor_3.write(0x04, duty)
        if speed > 0:
            motor_3.write(0x08, 0)
            motor_2.write(0x08, 1)
        else:
            motor_3.write(0x08, 1)
            motor_2.write(0x08, 0)

    def set_right_speed(self, speed):
        duty = int(self.size * (abs(speed) / 100))
        motor_1.write(0x04, duty)
        motor_0.write(0x04, duty)
        if speed > 0:
            motor_1.write(0x08, 0)
            motor_0.write(0x08, 1)
        else:
            motor_1.write(0x08, 1)
            motor_0.write(0x08, 0)

    # ---- 조향 폐루프 (data_collection 그대로) ----
    def control_motors(self):
        adc_raw = self.read_adc(spi0)
        mapped_resistance = self.map_value(
            adc_raw,
            self.resistance_most_right,
            self.resistance_most_left,
            -20, 20
        )
        # average between in_min, in_max = 1,925
        # ((2338 - 1925) * (20 - (-20)) / (2338 - 1512)) + (-20)
        # 20 + (-20) = 0

        # 조향 트림: 물리적 직진이 mapped=STEER_TRIM 이므로 목표에 오프셋
        target = self.steering_angle + STEER_TRIM
        self._dbg_target = target            # 디버그: 목표 조향각(트림포함)
        self._dbg_mapped = mapped_resistance # 디버그: 실제 바퀴 위치
        self._dbg_adc = adc_raw              # 디버그: ADC 원시값 (블랙박스 기록용)

        if abs(mapped_resistance - target) < 0.5: # 히스테리시스 방지
            self._dbg_cmd = "stay"
            self.stay()
        elif mapped_resistance > target: # 만약 바퀴 위치 > 목표 조향각이 양수 (목표보다 오른쪽으로 틀어져 있다)  -> 왼쪽으로 조향
            self._dbg_cmd = "left"
            self.left()
        else: # 바퀴 위치 < 목표 조향각 (목표보다 왼쪽으로 틀어져 있다) -> 오른쪽으로 조향
            self._dbg_cmd = "right"
            self.right()
        self.set_left_speed(self.left_speed)
        self.set_right_speed(self.right_speed)

    # ---- 제어 스레드: 100Hz로 폐루프 계속 (data_collection 키보드 스레드 역할) ----
    def control_loop(self):
        period = 1.0 / CONTROL_HZ
        while not self.stop_flag: 
            self.control_motors()
            time.sleep(period)

    # ---- 비전 결과 -> 조향 타겟(±20) ----
    def vision_to_target(self, center):
        if center is None:
            return 0.0
        offset = center - REF_X
        target = STEER_DIR * offset * STEER_GAIN
        return float(np.clip(target, -20, 20))

    # ---- 메인: 카메라/DPU 루프. 목표각만 갱신(모터 write 안 함) ----
    def run(self, camera_index=0):

        # 1. VideoCapture 인스턴스화. 목표 해상도 640 x 480
        cap = cv2.VideoCapture(camera_index)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        if not cap.isOpened():
            logger.critical("카메라를 열 수 없습니다.")
            return

        # 2. 블랙박스 기록 시작 (실시간 화면표시 대신 디스크에 기록)
        if RECORD_ENABLE:
            self.recorder = FlightRecorder()

        # 3. 출발: 구동 속도 인가 + 제어 스레드 시작
        self.left_speed = self.speed
        self.right_speed = self.speed
        ctrl = threading.Thread(target=self.control_loop, daemon=True)
        ctrl.start() # 100 Hz 주기로 조향 목표를 실제 제어 로직에 업데이트

        image_widget = widgets.Image(format='jpeg') # 노트북 이미지 표시용
        frame_i = 0
        try:
            while not self.stop_flag:
                
                # 4. VideoCapture 인스턴스에서 프레임 읽기
                ret, frame = cap.read()
                if not ret:
                    logger.critical("프레임을 읽을 수 없습니다.")
                    break

                # 5. 비전 파이프라인 실행: BEV -> ROI -> 256 리사이즈 -> DPU -> 차선검출 -> 각도
                # angle: 빨간 선(ref점 -> 검출점)이 수직선(직진)에서 기울어진 각도(도).
                #        0 = 직진, 음수 = 왼쪽, 양수 = 오른쪽. 연속(불연속 없음).
                #        center 범위 [0, 256] 기준 약 -42도 ~ +14도 (REF_X=200이라 비대칭).
                # center는 이번 프레임에서 오른쪽에 검출된 차선 YOLO 박스의 중심 x 좌표
                angle, vis = self.image_processor.process_frame(frame)
                center = self.image_processor.last_lane_center

                # 차선중심 EMA(Exponential Moving Average) 스무딩 -> 목표각 갱신 (제어 스레드가 이 값을 소비)
                if center is not None:
                    # if self._center_ema is None:
                    #     self._center_ema = float(center)
                    # else:
                    #     self._center_ema = CENTER_EMA * center + (1 - CENTER_EMA) * self._center_ema
                    # self.steering_angle = self.vision_to_target(self._center_ema)

                    # raw angle(도) 직접 사용 (디버깅용).
                    # 주의: 각도(도)와 조향 스케일(±20)은 단위가 다름. 중앙 부근 감도는
                    # 약 0.26/px로 vision_to_target(0.156/px)보다 ~1.7배 민감. 과민하면 배율 축소 고려.
                    self.steering_angle = float(np.clip(angle, -20, 20))

                else:
                    self.steering_angle = 0.0   # 미검출 -> 중립(직진)

                # 6. 블랙박스 기록: 이 프레임의 인식/판단/제어 상태를 한 줄로 남김
                if self.recorder is not None:
                    ip = self.image_processor
                    self.recorder.record(vis, {
                        "t": round(time.time(), 3),            # 절대 시각(초)
                        "i": frame_i,                          # 프레임 번호
                        "center": center,                      # [인식] 차선 중심 x (미검출이면 None)
                        "angle": None if center is None else round(float(angle), 2),  # [인식] 각도(도)
                        "steer": round(self.steering_angle, 2),# [판단] 목표 조향(±20, 트림 전)
                        "target": round(self._dbg_target, 2),  # [판단] 목표 조향(트림 포함) — 제어가 추종하는 값
                        "mapped": round(self._dbg_mapped, 2),  # [제어] 실제 바퀴 위치(±20)
                        "adc": self._dbg_adc,                  # [제어] ADC 원시값
                        "cmd": self._dbg_cmd,                  # [제어] 조향 명령 (left/stay/right)
                        "exec_ms": round(ip.last_exec_ms, 1),  # [인식] DPU 추론 시간
                        "boxes": ip.last_boxes,                # [인식] YOLO 박스 [y1,x1,y2,x2]
                        "scores": ip.last_scores,              # [인식] YOLO confidence
                    })

                # 7. 하트비트: 살아있는지 확인용으로 N프레임마다 한 줄만 출력 (wifi 스팸 방지)
                if frame_i % HEARTBEAT_EVERY == 0:
                    print(f"[STEER] i={frame_i} center={center} target={self._dbg_target:+.1f} "
                          f"mapped={self._dbg_mapped:+.1f} err={self._dbg_target - self._dbg_mapped:+.1f} "
                          f"cmd={self._dbg_cmd}")

                # 8. (옵션) 실시간 화면표시. 기본 꺼짐 — 프레임마다 JPEG 인코딩 + wifi 전송이
                #    주행 루프를 지연시켜 "관측이 대상을 바꾸는" 문제를 만들기 때문.
                frame_i += 1
                if DISPLAY_ENABLE and frame_i % DISPLAY_EVERY == 0:
                    ok, jpeg = cv2.imencode('.jpeg', vis)
                    if ok:
                        image_widget.value = jpeg.tobytes()
                        display(image_widget)
                        clear_output(wait=True)
        except KeyboardInterrupt:
            print("사용자 중지 (Ctrl+C)")
        finally:
            self.stop_flag = True
            ctrl.join(timeout=1.0)
            cap.release()
            if self.recorder is not None:
                self.recorder.close()
            self.cleanup()

    def cleanup(self):
        for motor in [motor_0, motor_1, motor_2, motor_3, motor_4, motor_5]:
            motor.write(0x00, 0)
            motor.write(0x04, 0)
            motor.write(0x08, 0)
        clear_output(wait=True)
        print("모터 정지 및 종료.")


print("RobotController(threaded control @%dHz, blackbox) ready" % CONTROL_HZ)


## ImageProcessor

In [ ]:
class ImageProcessor:
    # 비전 파이프라인: BEV(학습 전처리와 동일) -> ROI -> 256 리사이즈 -> DPU -> 차선검출 -> 각도
    def __init__(self):
        self.point_detection_height = 20   # 검출점 y (256 리사이즈 기준)
        self.reference_point_x = 200       # 각도 기준점 x (canonical image_processor.py 값)
        self.reference_point_y = 240
        self.last_lane_center = None       # 최근 검출된 차선 중심 x (조향 타겟 계산용)
        # 블랙박스 기록용: 최근 프레임의 인식 결과
        self.last_boxes = []               # YOLO 박스 [y1,x1,y2,x2] 리스트
        self.last_scores = []              # YOLO confidence 리스트
        self.last_exec_ms = 0.0            # DPU 추론 시간(ms)

    def roi_rectangle_below(self, img, cutting_idx):
        return img[cutting_idx:, :]

    def warpping(self, image, srcmat, dstmat):
        h, w = image.shape[0], image.shape[1]
        M = cv2.getPerspectiveTransform(srcmat, dstmat)
        minv = cv2.getPerspectiveTransform(dstmat, srcmat)
        warped = cv2.warpPerspective(image, M, (w, h))
        return warped, minv

    def bird_convert(self, img, srcmat, dstmat):
        srcmat = np.float32(srcmat)
        dstmat = np.float32(dstmat)
        warped, _ = self.warpping(img, srcmat, dstmat)
        return warped

    def calculate_angle(self, x1, y1, x2, y2):
        # 수직선(직진 방향) 기준 각도. 0 = 직진, 음수 = 왼쪽, 양수 = 오른쪽. 전 구간 연속.
        # 이미지 좌표는 y가 아래로 증가하므로 세로 거리는 y1 - y2 (기준점이 아래, 검출점이 위).
        # atan(slope) 방식은 수평축 기준이라 수직선 근처(±90도)에서 불연속이 생겨 atan2로 교체함.
        return math.degrees(math.atan2(x2 - x1, y1 - y2))

    def detect_lane_center_x(self, xyxy_results):
        # 가장 오른쪽 레이블(차선)의 중심 X (canonical과 동일 로직)
        rx_min = None; rx_max = None; rightmost = -float('inf')
        for box in xyxy_results:
            y1, x1, y2, x2 = box
            if x1 > rightmost:
                rightmost = x1
                rx_min = int(x1); rx_max = int(x2)
        if rx_min is not None:
            return (rx_min + rx_max) // 2
        return None

    def process_frame(self, img):
        h, w = img.shape[0], img.shape[1]
        # (1) BEV 변환 - 학습 전처리와 동일 (driving/image_processor.py 실측 좌표)
        src_mat = [[238, 316], [402, 313], [501, 476], [155, 476]]
        dst_mat = [[round(w * 0.3), 0], [round(w * 0.7), 0],
                   [round(w * 0.7), h], [round(w * 0.3), h]]
        bird_img = self.bird_convert(img, src_mat, dst_mat)

        # (2) ROI 하단 추출(cutting_idx=300, 캐노니컬과 동일) + 256 리사이즈
        roi_image = self.roi_rectangle_below(bird_img, cutting_idx=300)
        resized_img = cv2.resize(roi_image, (256, 256))

        # (3) 전처리 + DPU 추론
        image_size = resized_img.shape[:2]
        image_data = np.array(pre_process(resized_img, (256, 256)), dtype=np.float32)
        start_time = time.time()
        image[0, ...] = image_data.reshape(shapeIn[1:])
        job_id = dpu.execute_async(input_data, output_data)
        dpu.wait(job_id)
        end_time = time.time()
        self.last_exec_ms = (end_time - start_time) * 1000.0

        conv_out0 = np.reshape(output_data[0], shapeOut0)
        conv_out1 = np.reshape(output_data[1], shapeOut1)
        yolo_outputs = [conv_out0, conv_out1]
        boxes, scores, classes = evaluate(yolo_outputs, image_size, class_names, anchors, YOLO_THRESHOLD)

        # 블랙박스 기록용으로 인식 결과 보관 (JSON 직렬화 가능한 형태로)
        self.last_boxes = [[round(float(v), 1) for v in b] for b in boxes]
        self.last_scores = [round(float(s), 3) for s in scores]

        # 시각화(박스)
        for box in boxes:
            cv2.rectangle(resized_img, (int(box[1]), int(box[0])),
                          (int(box[3]), int(box[2])), (0, 255, 0), 2)

        # (4) 차선 중심 + 각도
        center = self.detect_lane_center_x(boxes) # 오른쪽 차선 YOLO 박스의 중심 X 좌표
        self.last_lane_center = center 
        if center is None:
            return 0.0, resized_img  # 새 각도 규약에서 0 = 직진(중립)

        # (ref_x, ref_y)와 (center, point_detection_height) 사이의 각도 계산.
        # 즉, 시각화 이미지에서 빨간색 선이 수직선(직진)으로부터 기울어진 각도.
        # 0 = 직진(center == ref_x), 음수 = 왼쪽, 양수 = 오른쪽. 불연속 없음.
        # 예: (200, 240) vs (190, 20) -> atan2(190-200, 240-20) = atan2(-10, 220) = -2.60도
        # center 범위 [0, 256] 기준: 약 -42.3도(맨 왼쪽) ~ 0(중앙) ~ +14.3도(맨 오른쪽)
        angle = self.calculate_angle(self.reference_point_x, self.reference_point_y,
                                     center, self.point_detection_height)

        # 시각화(라인/중심점)
        # 1. (ref_x, ref_y) 부터 (center, point_detection_height) 까지 빨간색 선을 긋는다.
        # 2. (center, point_detection_height)에 노란색 점을 찍는다.
        cv2.line(resized_img, (self.reference_point_x, self.reference_point_y),
                 (center, self.point_detection_height), (0, 0, 255), 3)
        cv2.circle(resized_img, (center, self.point_detection_height), 6, (0, 255, 255), -1)

        return angle, resized_img


print("ImageProcessor(BEV, blackbox) ready")


## 수동 조향 방향 확인 (바퀴 띄우고!)

In [ ]:
# ===== 조향 방향 확인: left()/right()가 실제 원하는 방향으로 도는지 눈으로 확인 =====
# 주의: 반드시 바퀴를 지면에서 띄운 상태로! 구동 모터는 건들지 않고 조향만 테스트.
# left()/right()를 직접 호출(폐루프 X, 개루프). 만약 방향이 반대면 배선/모터매핑 문제.
import time

tester = RobotController()   # 모터 초기화(전부 중립)

def hold(action, name, secs=1.0):
    print(f"[{name}] {secs}s ...")
    t0 = time.time()
    while time.time() - t0 < secs:
        action()
        time.sleep(0.02)
    tester.stay()
    time.sleep(0.6)

try:
    hold(tester.right, "RIGHT -> 조향이 오른쪽으로 가야 정상")
    hold(tester.left,  "LEFT  -> 조향이 왼쪽으로 가야 정상")
finally:
    tester.stay()
    tester.cleanup()
    print("조향 방향 테스트 종료")


## ADC 조향 캘리브레이션 확인 (바퀴 띄우고!)

In [ ]:
# ===== 직진일 때 ADC 값 확인: 폐루프의 '직진'(target=0)이 실제 직진과 맞나 =====
# 사용법: 바퀴를 손으로 (1)정면직진 (2)풀좌 (3)풀우 에 두고 각각 이 셀 실행해 값 읽기.
# mapped=0 은 ADC 1925(=1512·2338 중점). 물리적 직진이 여기서 벗어나면 그게 우측 쏠림 원인.
import time
def _adc(spi):
    r = spi.xfer2([0x00, 0x00]); return ((r[0] & 0x0F) << 8) | r[1]
def _mapped(v, right=1512, left=2338):
    if v <= right: return 20.0
    if v >= left:  return -20.0
    return (left - v) * 40.0 / (left - right) - 20.0
vals = []
print('3초간 ADC 읽는 중 (조향 위치 고정하고 값 보기)...')
for _ in range(30):
    v = _adc(spi0); vals.append(v)
    print(f'ADC={v:4d}  mapped={_mapped(v):+5.1f}')
    time.sleep(0.1)
import statistics
med = int(statistics.median(vals))
print(f'--- 중앙값 ADC={med}  mapped={_mapped(med):+.1f}  (직진이면 mapped≈0, ADC≈1925 기대) ---')


## Driving!

In [ ]:
# ===== 실행 (블랙박스 기록 모드) =====
# 주의: 첫 테스트는 반드시 바퀴를 지면에서 띄운 상태로!
#  - 좌/우 바퀴가 같은 방향(전진)으로 도는지
#  - 차선이 화면 오른쪽이면 우조향인지 (반대면 위 셀의 STEER_DIR 을 -1 로)
#
# 주행이 끝나면(Ctrl+C 또는 셀 중지) 기록이 자동으로 닫히고 run_* 폴더 경로가 출력됨.
# 맥에서 가져와서 replay_viewer.html 로 분석:
#   scp -r xilinx@<보드IP>:/home/xilinx/jupyter_notebooks/KGIC/driveCode/logs/run_* ~/AIchip/logs/
controller = RobotController()
controller.run(camera_index=0)